# Credit Card Fraud Detection using Machine Learning on Imbalanced Data
### A Complete Step-by-Step Practical Guide for Beginners & Freshers

---

### What is this project about?
Credit card fraud occurs when an unauthorized person gains access to someone's credit card information to make purchases or transactions. 
Because fraudulent transactions happen rarely compared to the millions of legitimate purchases made every day, the data is **heavily imbalanced** (less than 1% fraud).

In this notebook, we walk through the complete machine learning lifecycle:
1. **Understand the Problem & The Accuracy Paradox:** Why 99.4% accuracy can mean a completely useless model.
2. **Exploratory Data Analysis (EDA):** Uncovering fraud patterns in transaction amount, spending category, and time of day.
3. **Feature Engineering:** Creating domain features (distance in km, transaction hour, night-time flag, cardholder age).
4. **Handling Imbalance:** Using cost-sensitive learning (`class_weight='balanced'`) and evaluation metrics (PR-AUC, Recall, Precision).
5. **Model Building & Comparison:** Logistic Regression, Decision Tree, and Gradient Boosted Trees.
6. **Evaluation & Threshold Tuning:** Optimizing decision boundaries for maximum fraud detection.
7. **Final Test on Unseen Data (`fraudTest.csv`)** and Model Persistence.


## Step 1: Import Necessary Libraries
We import standard data science and machine learning packages:
- `pandas` & `numpy` for data manipulation
- `matplotlib` & `seaborn` for data visualization
- `scikit-learn` for machine learning algorithms, preprocessing, and evaluation metrics


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve
)
import joblib

# Set styling for plots
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries imported successfully!')


## Step 2: Loading & Inspecting the Dataset
The dataset provided contains:
- `fraudTrain.csv`: Simulated credit card transactions for training
- `fraudTest.csv`: Separate transaction dataset for testing

Let's load `fraudTrain.csv` and inspect its shape and columns.
*(Note: To keep exploratory execution fast and responsive, we can inspect a representative sample of 250,000 transactions, or the full dataset).*


In [ ]:
# Load dataset (using 250,000 rows for fast and interactive experimentation)
DATA_PATH = 'fraudTrain.csv'
print(f'Loading data from {DATA_PATH}...')

df = pd.read_csv(DATA_PATH, nrows=250000)

print(f'Dataset Shape: {df.shape[0]:,} rows and {df.shape[1]} columns')
df.head(3)


Let's inspect the columns, data types, and check for missing/null values.


In [ ]:
# Check data types and missing values
print('--- Missing Values Check ---')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found! The dataset is clean.')

print('
--- Column Data Types ---')
print(df.dtypes)


## Step 3: Exploratory Data Analysis (EDA)

### 3.1 Understanding the Class Imbalance
Let's see how many transactions are legitimate (`is_fraud = 0`) vs fraudulent (`is_fraud = 1`).


In [ ]:
# Class distribution
fraud_counts = df['is_fraud'].value_counts()
fraud_pct = df['is_fraud'].value_counts(normalize=True) * 100

print('Class Counts:')
print(f'Legitimate (0): {fraud_counts[0]:,} ({fraud_pct[0]:.2f}%)')
print(f'Fraudulent (1): {fraud_counts[1]:,} ({fraud_pct[1]:.2f}%)')
print(f'Ratio: Approximately 1 fraud in every {int(fraud_counts[0]/fraud_counts[1])} transactions')

# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.barplot(x=['Legitimate (0)', 'Fraudulent (1)'], y=fraud_counts.values, ax=axes[0], palette=['#2b5c8f', '#d9534f'])
axes[0].set_title('Transaction Count by Class (Notice the Extreme Imbalance!)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(fraud_counts.values, labels=['Legitimate (99.4%)', 'Fraud (0.6%)'], colors=['#2b5c8f', '#d9534f'],
            autopct='%1.2f%%', startangle=90, explode=(0, 0.2))
axes[1].set_title('Percentage Share of Fraud vs Normal', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


### 3.2 Transaction Amount: Normal vs. Fraudulent
Do fraudsters steal higher amounts on average? Let's compare the statistics.


In [ ]:
# Compare transaction amounts
print('--- Legitimate Transactions Amount Summary ---')
print(df[df['is_fraud'] == 0]['amt'].describe())

print('
--- Fraudulent Transactions Amount Summary ---')
print(df[df['is_fraud'] == 1]['amt'].describe())

# Visualizing Amount Distribution
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(x='is_fraud', y='amt', data=df, palette=['#2b5c8f', '#d9534f'], ax=ax, showfliers=False)
ax.set_xticklabels(['Legitimate (0)', 'Fraudulent (1)'])
ax.set_title('Transaction Amount by Class (Excluding Outliers)', fontsize=12, fontweight='bold')
ax.set_ylabel('Amount ($)')
plt.show()


**Key Insight from EDA:**
- Normal transactions have a median amount of **~$47**.
- Fraudulent transactions have a median amount of **~$346** and a mean of **~$509**! Fraudulent activity clearly skews toward higher-value transactions.


### 3.3 Which spending categories have the highest fraud rate?


In [ ]:
# Category fraud analysis
cat_analysis = df.groupby('category').agg(
    total_tx=('is_fraud', 'count'),
    fraud_tx=('is_fraud', 'sum'),
    fraud_rate=('is_fraud', 'mean')
).sort_values('fraud_rate', ascending=False).reset_index()

cat_analysis['fraud_rate_pct'] = cat_analysis['fraud_rate'] * 100

plt.figure(figsize=(12, 5))
sns.barplot(data=cat_analysis, x='category', y='fraud_rate_pct', palette='Reds_r')
plt.title('Fraud Rate (%) Across Transaction Categories', fontsize=13, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Fraud Rate (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

cat_analysis[['category', 'total_tx', 'fraud_tx', 'fraud_rate_pct']]


**Key Insight:** Online shopping (`shopping_net`), grocery stores (`grocery_pos`), and miscellaneous internet purchases (`misc_net`) show significantly higher fraud rates!


### 3.4 At what time of the day does fraud occur?
Let's extract the transaction hour and check the fraud rate hour-by-hour.


In [ ]:
# Extract transaction hour
df['trans_hour'] = pd.to_datetime(df['trans_date_trans_time']).dt.hour

hourly_fraud = df.groupby('trans_hour')['is_fraud'].agg(['count', 'mean']).reset_index()
hourly_fraud['fraud_pct'] = hourly_fraud['mean'] * 100

plt.figure(figsize=(12, 4))
sns.lineplot(data=hourly_fraud, x='trans_hour', y='fraud_pct', marker='o', color='#d9534f', linewidth=2.5)
plt.title('Fraud Probability by Hour of the Day (24-Hour Clock)', fontsize=13, fontweight='bold')
plt.xlabel('Hour of Day (0 to 23)')
plt.ylabel('Fraud Rate (%)')
plt.xticks(range(0, 24))
plt.axvspan(22, 23, color='yellow', alpha=0.3, label='Peak Fraud Hours (10 PM - 3 AM)')
plt.axvspan(0, 3, color='yellow', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


**Key Insight:** Fraud spikes dramatically between **10 PM and 3 AM** (late at night when cardholders are asleep and less likely to immediately notice bank SMS alerts)!


## Step 4: Feature Engineering & Preprocessing

Machine learning algorithms need meaningful numerical features. We will create:
1. `trans_hour`: Hour of transaction (0-23).
2. `trans_day_of_week`: Day of week (0-6).
3. `is_night`: Binary flag (1 if between 10 PM and 4 AM, 0 otherwise).
4. `age`: Cardholder's age computed from Date of Birth (`dob`).
5. `distance_km`: Great-circle distance between cardholder residence and merchant location using the **Haversine formula**.
6. `log_amt`: Log-transformed transaction amount to reduce skewness.

We also drop personal identifier columns (`first`, `last`, `cc_num`, `trans_num`, `street`) that would lead to overfitting and leakage.


In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculates great-circle distance in kilometers between two GPS coordinates."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c  # Earth radius in km

def engineer_features(data):
    """Creates features and cleans DataFrame."""
    data = data.copy()
    
    # 1. Date & Time features
    dt = pd.to_datetime(data['trans_date_trans_time'])
    data['trans_hour'] = dt.dt.hour
    data['trans_day_of_week'] = dt.dt.dayofweek
    data['is_night'] = data['trans_hour'].apply(lambda h: 1 if (h >= 22 or h <= 4) else 0)
    
    # 2. Age feature
    dob_dt = pd.to_datetime(data['dob'])
    data['age'] = 2020 - dob_dt.dt.year
    
    # 3. Distance between cardholder and merchant
    data['distance_km'] = haversine_distance(
        data['lat'], data['long'], data['merch_lat'], data['merch_long']
    )
    
    # 4. Log Amount
    data['log_amt'] = np.log1p(data['amt'])
    
    # 5. Drop unnecessary identifier columns
    drop_cols = [
        'Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'first', 'last',
        'street', 'city', 'state', 'zip', 'lat', 'long', 'job', 'dob',
        'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'merchant'
    ]
    data = data.drop(columns=[col for col in drop_cols if col in data.columns])
    
    return data

# Process data
processed_df = engineer_features(df)
print(f'Processed Features Shape: {processed_df.shape}')
processed_df.head(3)


### Encoding Categorical Variables
We encode `category` and `gender` into numerical values using `LabelEncoder`.


In [ ]:
# Encode categorical columns
label_encoders = {}
for col in ['category', 'gender']:
    le = LabelEncoder()
    processed_df[col] = le.fit_transform(processed_df[col].astype(str))
    label_encoders[col] = le

print('Features ready for modeling:')
print(processed_df.columns.tolist())


## Step 5: Stratified Train-Validation Split
Because the positive class (fraud) is less than 1%, we **must** use `stratify=y`. 
This guarantees that both the training and validation sets have the exact same percentage of fraud transactions.


In [ ]:
# Separate features (X) and target (y)
X = processed_df.drop(columns=['is_fraud'])
y = processed_df['is_fraud']

# Stratified split: 80% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'X_train shape: {X_train.shape}, Fraud cases in train: {y_train.sum():,} ({y_train.mean()*100:.2f}%)')
print(f'X_val shape:   {X_val.shape}, Fraud cases in val:   {y_val.sum():,} ({y_val.mean()*100:.2f}%)')


## Step 6: Demonstrating The "Accuracy Paradox"
What happens if a beginner builds a naive model that predicts "0" (Not Fraud) for every transaction?


In [ ]:
# Naive baseline: Always predict 0 (No Fraud)
y_naive_pred = np.zeros(len(y_val))

print('=== NAIVE MODEL (ALWAYS PREDICTS NOT FRAUD) ===')
print(f'Accuracy:        {accuracy_score(y_val, y_naive_pred)*100:.2f}%  <-- Looks amazing!')
print(f'Recall (Fraud):  {recall_score(y_val, y_naive_pred)*100:.2f}%   <-- Caught ZERO fraud!')
print(f'Precision:       {precision_score(y_val, y_naive_pred, zero_division=0)*100:.2f}%')
print(f'F1-Score:        {f1_score(y_val, y_naive_pred, zero_division=0)*100:.2f}%')


**Takeaway:** A naive model gets **>99.4% accuracy**, but fails to catch even a single fraud! 
This is why in fraud detection, **Accuracy is meaningless** and our key metrics must be:
- **Recall:** How many of the actual fraud cases did we catch?
- **Precision:** When we raise a fraud alert, how often is it real?
- **PR-AUC (Precision-Recall AUC):** The overall trade-off between precision and recall across all thresholds.


## Step 7: Training Machine Learning Models
We compare three distinct models:
1. **Logistic Regression (Standard vs Balanced):** Demonstrates how `class_weight="balanced"` penalizes fraud errors.
2. **Decision Tree Classifier:** Rule-based non-linear tree model.
3. **HistGradientBoostingClassifier:** Fast, state-of-the-art gradient boosted decision tree ensemble with balanced weighting.


In [ ]:
# Evaluation helper function
results = []

def record_evaluation(name, model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1] if hasattr(model, 'predict_proba') else y_pred
    
    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_eval, y_prob)
    pr_auc = average_precision_score(y_eval, y_prob)
    
    res = {
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'PR-AUC': pr_auc,
        'y_pred': y_pred,
        'y_prob': y_prob
    }
    results.append(res)
    
    print(f'--- {name} ---')
    print(f'Accuracy: {acc:.4f} | Recall: {rec:.4f} | Precision: {prec:.4f} | PR-AUC: {pr_auc:.4f}')
    return res


### Model 1: Logistic Regression (Standard vs Balanced)


In [ ]:
from sklearn.pipeline import Pipeline

# 1A. Standard Logistic Regression (unweighted)
lr_std = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, max_iter=500))
])
lr_std.fit(X_train, y_train)
res_lr_std = record_evaluation('Logistic Regression (Standard)', lr_std, X_val, y_val)

# 1B. Balanced Logistic Regression (cost-sensitive)
lr_balanced = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', random_state=42, max_iter=500))
])
lr_balanced.fit(X_train, y_train)
res_lr_bal = record_evaluation('Logistic Regression (Balanced)', lr_balanced, X_val, y_val)


### Model 2: Decision Tree Classifier


In [ ]:
dt_model = DecisionTreeClassifier(
    max_depth=8,
    class_weight='balanced',
    random_state=42
)
dt_model.fit(X_train, y_train)
res_dt = record_evaluation('Decision Tree (Balanced)', dt_model, X_val, y_val)


### Model 3: HistGradientBoostingClassifier (Ensemble Boosted Trees)


In [ ]:
hgb_model = HistGradientBoostingClassifier(
    max_iter=100,
    class_weight='balanced',
    random_state=42
)
hgb_model.fit(X_train, y_train)
res_hgb = record_evaluation('HistGradientBoosting (Balanced)', hgb_model, X_val, y_val)


## Step 8: Model Evaluation & Comparison
Let's aggregate the results and visualize the trade-offs.


In [ ]:
# Create comparison DataFrame
comparison_rows = []
for r in results:
    comparison_rows.append({
        'Model': r['Model'],
        'Accuracy': f"{r['Accuracy']*100:.2f}%",
        'Precision': f"{r['Precision']*100:.2f}%",
        'Recall (Fraud)': f"{r['Recall']*100:.2f}%",
        'F1-Score': f"{r['F1-Score']:.4f}",
        'ROC-AUC': f"{r['ROC-AUC']:.4f}",
        'PR-AUC': f"{r['PR-AUC']:.4f}"
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


### Visualizing Confusion Matrices
Let's see how many frauds each model detected vs missed.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
eval_models = [res_lr_bal, res_dt, res_hgb]

for ax, res in zip(axes, eval_models):
    cm = confusion_matrix(y_val, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit (0)', 'Fraud (1)'],
                yticklabels=['Legit (0)', 'Fraud (1)'])
    ax.set_title(f"{res['Model']}\nCaught: {cm[1][1]} / {cm[1][0]+cm[1][1]} frauds", fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.tight_layout()
plt.show()


### Precision-Recall Curve Comparison
The Precision-Recall curve is the most informative plot for imbalanced classification:


In [ ]:
plt.figure(figsize=(9, 6))

for res in eval_models:
    precision, recall, _ = precision_recall_curve(y_val, res['y_prob'])
    plt.plot(recall, precision, label=f"{res['Model']} (PR-AUC = {res['PR-AUC']:.3f})")

plt.xlabel('Recall (Fraud Detection Rate)', fontsize=11)
plt.ylabel('Precision (True Fraud / Total Flagged)', fontsize=11)
plt.title('Precision-Recall Curves (Higher is Better)', fontsize=13, fontweight='bold')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()


## Step 9: Decision Threshold Tuning
By default, models classify a transaction as fraud if the probability is $\ge 0.50$.
In banking, we often lower the threshold (e.g. to $0.20$) to catch even more fraud, accepting a small number of extra manual verification checks.


In [ ]:
# Analyze different decision thresholds for the best model (HistGradientBoosting)
best_probs = res_hgb['y_prob']
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
tuning_results = []

for t in thresholds:
    t_preds = (best_probs >= t).astype(int)
    cm = confusion_matrix(y_val, t_preds)
    tuning_results.append({
        'Threshold': t,
        'Recall (Fraud %)': f"{recall_score(y_val, t_preds)*100:.2f}%",
        'Precision (%)': f"{precision_score(y_val, t_preds, zero_division=0)*100:.2f}%",
        'F1-Score': round(f1_score(y_val, t_preds, zero_division=0), 4),
        'Frauds Caught': cm[1, 1],
        'False Alarms': cm[0, 1]
    })

pd.DataFrame(tuning_results)


## Step 10: Final Evaluation on Unseen Test Dataset (`fraudTest.csv`) & Model Saving
Now we test our best model on the independent `fraudTest.csv` dataset.


In [ ]:
TEST_PATH = 'fraudTest.csv'
print(f'Loading unseen test dataset from {TEST_PATH}...')
test_df = pd.read_csv(TEST_PATH, nrows=100000)

# Apply same feature engineering
processed_test = engineer_features(test_df)

# Apply saved label encoders
for col in ['category', 'gender']:
    le = label_encoders[col]
    processed_test[col] = processed_test[col].astype(str).map(
        lambda s: le.transform([s])[0] if s in le.classes_ else -1
    )

X_test = processed_test.drop(columns=['is_fraud'])
y_test = processed_test['is_fraud']

# Evaluate best model
final_metrics = record_evaluation('Final Test Evaluation (HistGradientBoosting)', hgb_model, X_test, y_test)


### Saving the Model & Label Encoders
We serialize the trained model and encoders so they can be loaded for real-time inference or deployment.


In [ ]:
model_artifacts = {
    'model': hgb_model,
    'label_encoders': label_encoders,
    'features': X.columns.tolist()
}

os.makedirs('models', exist_ok=True)
model_path = os.path.join('models', 'fraud_detection_model.joblib')
joblib.dump(model_artifacts, model_path)
print(f'Model and artifacts saved successfully to: {model_path}')


## Step 11: Summary & Key Takeaways for Freshers

1. **Accuracy is Deceptive on Imbalanced Data:** A model predicting 100% legitimate transactions gets 99.4% accuracy but detects zero fraud. Always use **Recall**, **Precision**, and **PR-AUC**.
2. **Cost-Sensitive Learning Works:** Setting `class_weight='balanced'` allows models like Logistic Regression and Gradient Boosted Trees to focus on the rare positive class without distorting feature relationships.
3. **Domain Features Drive Performance:**
   - **Transaction Amount:** Fraudsters target higher average values ($346 median vs $47).
   - **Time of Day:** Fraud spikes significantly between 10 PM and 3 AM.
   - **High-Risk Categories:** Online shopping and internet transactions are more prone to fraud.
4. **Threshold Tuning Gives Business Control:** Adjusting the classification threshold lets financial institutions choose the right balance between catching fraud and minimizing customer inconvenience.
